# **ILL392**: Build Agentic AI Postgres App with Azure HorizonDB

## Notebook 1 - Data Setup

In [1]:
# Standard library
import os
import time

# Database driver
import psycopg

# Azure OpenAI client
from openai import AzureOpenAI

# Environment variable loading
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_KEY        = os.environ["AZURE_OPENAI_KEY"]
AZURE_EMBED_DEPLOYMENT  = os.environ["AZURE_EMBED_DEPLOYMENT"]
AZURE_API_VERSION       = os.environ["AZURE_API_VERSION"]

DB_CONFIG = {
    "host":     os.environ["AZURE_PG_HOST"],
    "dbname":   os.environ["AZURE_PG_NAME"],
    "user":     os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port":     os.environ["AZURE_PG_PORT"],
    "sslmode":  os.environ.get("AZURE_PG_SSLMODE", "require"),
}

print(AZURE_OPENAI_DEPLOYMENT)
print(AZURE_EMBED_DEPLOYMENT)
print(AZURE_API_VERSION)


In [ ]:
# Create Extensions Needed for Lab
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS azure_ai CASCADE;")
    print("AZURE_AI extension ready.")

    cur.execute("CREATE EXTENSION IF NOT EXISTS vector CASCADE;")
    print("VECTOR extension ready.")

    cur.execute("CREATE EXTENSION IF NOT EXISTS age CASCADE;")
    print("AGE extension ready.")

    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_diskann CASCADE;")
    print("DISKANN extension ready.")

    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_textsearch;")
    print("PG_TEXTSEARCH extension ready.")

conn.close()
print("Done. Extensions Created.")


In [ ]:
# Register Azure OpenAI deployments in the azure_ai model registry
CHAT_MODEL_ALIAS  = "lab-chat"
EMBED_MODEL_ALIAS = "lab-embedding"

conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Remove any prior registrations so this cell is safely re-runnable
    for alias in (CHAT_MODEL_ALIAS, EMBED_MODEL_ALIAS):
        try:
            cur.execute("SELECT model_registry.model_remove(%s);", (alias,))
        except Exception:
            pass

    # Register the chat-completion deployment
    cur.execute(
        """
        SELECT model_registry.model_add(
            %s,                 -- model alias
            %s,                 -- endpoint URL
            %s,                 -- deployment name
            %s,                 -- model name
            %s,                 -- API version
            'subscription-key', -- auth type
            %s                  -- endpoint key
        );
        """,
        (
            CHAT_MODEL_ALIAS,
            AZURE_OPENAI_ENDPOINT,
            AZURE_OPENAI_DEPLOYMENT,
            AZURE_OPENAI_DEPLOYMENT,
            AZURE_API_VERSION,
            AZURE_OPENAI_KEY,
        ),
    )
    print(f"Registered chat model alias:      {CHAT_MODEL_ALIAS}")

    # Register the embedding deployment
    cur.execute(
        """
        SELECT model_registry.model_add(
            %s, %s, %s, %s, %s, 'subscription-key', %s
        );
        """,
        (
            EMBED_MODEL_ALIAS,
            AZURE_OPENAI_ENDPOINT,
            AZURE_EMBED_DEPLOYMENT,
            AZURE_EMBED_DEPLOYMENT,
            AZURE_API_VERSION,
            AZURE_OPENAI_KEY,
        ),
    )
    print(f"Registered embedding model alias: {EMBED_MODEL_ALIAS}")

    # Verify the registrations
    cur.execute(
        """
        SELECT alias, endpoint, deployment_name, model_name, api_version, auth_type        
        FROM model_registry.model_list_all()        
        ORDER BY alias;
        """        
    )
    for row in cur.fetchall():
        print(row)

conn.close()
print("Done. azure_ai models registered from environment variables.")


In [ ]:
# Initialize dataset: create tables and load cases from CSV
csv_path = os.path.join(os.path.dirname(os.getcwd()), "Dataset", "cases.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.getcwd(), "Dataset", "cases.csv")
print(f"CSV path: {csv_path}")

conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Drop existing tables
    cur.execute("DROP TABLE IF EXISTS cases;")
    cur.execute("DROP TABLE IF EXISTS temp_cases;")

    # Create the cases table
    cur.execute("""
        CREATE TABLE cases(
            id SERIAL PRIMARY KEY,
            name TEXT,
            decision_date DATE,
            court_level TEXT,
            opinion TEXT
        );
    """)

    # Create temp staging table and load CSV
    cur.execute("CREATE TABLE temp_cases(data jsonb);")

    with open(csv_path, "r", encoding="utf-8") as f:
        with cur.copy("COPY temp_cases (data) FROM STDIN WITH (FORMAT csv, HEADER true)") as copy:
            while chunk := f.read(8192):
                copy.write(chunk)

    print("CSV loaded into temp_cases.")

    # Insert parsed data into cases table
    cur.execute("""
        INSERT INTO cases
        SELECT
            (data#>>'{id}')::int AS id,
            (data#>>'{name_abbreviation}')::text AS name,
            (data#>>'{decision_date}')::date AS decision_date,
            (data#>>'{court,name}')::text AS court_level,
            array_to_string(
                ARRAY(SELECT jsonb_path_query(data, '$.casebody.opinions[*].text')),
                ', '
            ) AS opinion
        FROM temp_cases;
    """)

    cur.execute("SELECT count(*) FROM cases;")
    count = cur.fetchone()[0]
    print(f"Done. {count} cases loaded into the cases table.")

conn.close()

In [ ]:
# Test query: select top 5 rows from cases
conn = psycopg.connect(**DB_CONFIG)
with conn.cursor() as cur:
    cur.execute("SELECT * FROM cases LIMIT 5;")
    rows = cur.fetchall()
    for row in rows:
        print(row)
conn.close()

In [ ]:
# Add the vector column to the cases table
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute("ALTER TABLE cases ADD COLUMN IF NOT EXISTS opinions_vector vector(1536);")    
conn.close()

print("opinions_vector column added.")

In [ ]:
BATCH_SIZE = 20
CLEAR_VECTORS_BEFORE_RUN = True  # Set to False to skip clearing existing vectors

# Azure OpenAI client for embeddings
embed_client = AzureOpenAI(
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

# Connect and fetch rows that still need embeddings
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

# Optionally clear all existing vectors
if CLEAR_VECTORS_BEFORE_RUN:
    with conn.cursor() as cur:
        cur.execute("UPDATE cases SET opinions_vector = NULL")
    print("Cleared all existing vectors.")

start_time = time.time()
print(f"Start time: {time.strftime('%H:%M:%S', time.localtime(start_time))}")

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM cases WHERE opinions_vector IS NULL")
    total_rows = cur.fetchone()[0]
    print(f"Total rows to process: {total_rows}")

    cur.execute("SELECT id, name, opinion FROM cases WHERE opinions_vector IS NULL ORDER BY id")
    rows = cur.fetchall()

processed = 0
for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i : i + BATCH_SIZE]
    texts = [(name or "") + " " + (opinion or "")[:8000] for _id, name, opinion in batch]

    # Call Azure OpenAI embeddings
    response = embed_client.embeddings.create(
        input=texts,
        model=AZURE_EMBED_DEPLOYMENT,
    )

    # Update each row with its embedding
    with conn.cursor() as cur:
        for j, item in enumerate(response.data):
            vector_str = "[" + ",".join(str(v) for v in item.embedding) + "]"
            cur.execute(
                "UPDATE cases SET opinions_vector = %s::vector WHERE id = %s",
                (vector_str, batch[j][0]),
            )

    processed += len(batch)
    pct = round(100.0 * processed / total_rows, 1)
    print(f"Progress: {processed} / {total_rows} ({pct}%)")

conn.close()

end_time = time.time()
duration = end_time - start_time
print(f"\nEnd time: {time.strftime('%H:%M:%S', time.localtime(end_time))}")
print(f"Total duration: {int(duration // 60)}m {duration % 60:.1f}s")
print(f"Done. {processed} rows processed.")

In [ ]:
# Build the case citation graph using Apache AGE
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # ------------------------------------------------------------------------
    # 0) Make sure ag_catalog is on the search path *now*, before we CREATE
    #    the helper functions below. PL/pgSQL resolves the `agtype` type at
    #    function-create time, so without this we'd get:
    #        UndefinedObject: type "agtype" does not exist
    #    (AGE itself was already installed in Part 2.4.)
    # ------------------------------------------------------------------------
    cur.execute("CREATE EXTENSION IF NOT EXISTS age CASCADE;")
    cur.execute("SET search_path = public, ag_catalog, \"$user\";")

    # ------------------------------------------------------------------------
    # 1) Helper functions: thin SQL wrappers around AGE's `cypher(...)` call.
    #
    # AGE exposes Cypher through cypher(graph_name, query_text, params_agtype).
    # The third (params) argument is a quirk: AGE *parses* the call and only
    # accepts it if the third argument is a literal parameter reference
    # (e.g. `$1`). Passing `agtype_build_map(...)` inline raises:
    #
    #     "third argument of cypher function must be a parameter"
    #
    # The workaround used below: build the agtype map into a PL/pgSQL
    # variable, then run the cypher() call through `EXECUTE ... USING props`
    # so PL/pgSQL substitutes `$1` as a real bind parameter. This is also
    # the safest way to ship arbitrary string values (quotes, newlines,
    # unicode) into Cypher without any manual escaping.
    # ------------------------------------------------------------------------

    # 1a) Create a single :case node with all of its properties.
    cur.execute("""
        CREATE OR REPLACE FUNCTION create_case_in_case_graph(
            case_id       text,
            name          text,
            decision_date text,
            court_level   text,
            opinion       text
        )
        RETURNS void
        LANGUAGE plpgsql
        VOLATILE
        AS $BODY$
        DECLARE
            props ag_catalog.agtype;
        BEGIN
            -- AGE's cypher() and agtype_build_map() live in ag_catalog; put
            -- them on the search path for the duration of this function call.
            SET search_path TO ag_catalog;

            -- Build the parameter map. Each key here is referenced as
            -- $key inside the Cypher text below.
            props := agtype_build_map(
                'case_id',       case_id,
                'name',          name,
                'decision_date', decision_date,
                'court_level',   court_level,
                'opinion',       opinion
            );

            -- EXECUTE ... USING binds props as $1, which is what AGE
            -- requires for the third cypher() argument.
            EXECUTE $sql$
                SELECT * FROM cypher(
                    'case_graph',
                    $cy$
                        CREATE (:case {
                            case_id:       $case_id,
                            name:          $name,
                            decision_date: $decision_date,
                            court_level:   $court_level,
                            opinion:       $opinion
                        })
                    $cy$,
                    $1
                ) AS (a agtype);
            $sql$ USING props;
        END
        $BODY$;
    """)

    # 1b) Create a directed :REF edge from one case node to another.
    #     Same EXECUTE ... USING pattern as 1a.
    cur.execute("""
        CREATE OR REPLACE FUNCTION create_case_link_in_case_graph(id_from text, id_to text)
        RETURNS void
        LANGUAGE plpgsql
        VOLATILE
        AS $BODY$
        DECLARE
            props ag_catalog.agtype;
        BEGIN
            SET search_path TO ag_catalog;

            props := agtype_build_map('id_from', id_from, 'id_to', id_to);

            EXECUTE $sql$
                SELECT * FROM cypher(
                    'case_graph',
                    $cy$
                        MATCH (a:case), (b:case)
                        WHERE a.case_id = $id_from AND b.case_id = $id_to
                        CREATE (a)-[e:REF]->(b)
                        RETURN e
                    $cy$,
                    $1
                ) AS (a agtype);
            $sql$ USING props;
        END
        $BODY$;
    """)
    print("Helper functions created.")

    # ------------------------------------------------------------------------
    # 2) Drop any previous case_graph so this cell is safely re-runnable.
    #
    #    drop_graph() raises an error if the graph doesn't exist, so we guard
    #    it with a check against ag_catalog.ag_graph (AGE's metadata table).
    #    The second arg `true` = cascade (also drop the graph's backing tables).
    # ------------------------------------------------------------------------
    cur.execute("""
        DO $$
        BEGIN
            IF EXISTS (SELECT 1 FROM ag_catalog.ag_graph WHERE name = 'case_graph') THEN
                PERFORM ag_catalog.drop_graph('case_graph', true);
            END IF;
        END $$;
    """)
    print("Old case_graph dropped (if it existed).")

    # ------------------------------------------------------------------------
    # 3) Create a fresh, empty graph named 'case_graph'.
    # ------------------------------------------------------------------------
    cur.execute("SELECT create_graph('case_graph');")
    print("case_graph created.")

    # ------------------------------------------------------------------------
    # 4) Populate the nodes.
    #
    #    Pull node properties from the typed `cases` table (already parsed
    #    out of the raw JSON in Part 2.5). `id::text` matches the case_id
    #    string the citation edges will reference below.
    # ------------------------------------------------------------------------
    cur.execute("""
        SELECT create_case_in_case_graph(
            id::text,
            name,
            decision_date::text,
            court_level,
            opinion
        )
        FROM public.cases;
    """)
    print("Graph nodes created.")

    # ------------------------------------------------------------------------
    # 5) Sanity check: count the nodes back out via Cypher.
    #    This proves both the graph and the cypher() function are healthy
    #    before we spend time creating edges.
    # ------------------------------------------------------------------------
    cur.execute("""
        SELECT * FROM cypher('case_graph', $$
            MATCH (n)
            RETURN COUNT(n.case_id)
        $$) AS (case_id TEXT);
    """)
    node_count = cur.fetchone()[0]
    print(f"Node count: {node_count}")

    # ------------------------------------------------------------------------
    # 6) Build the citation edges.
    #
    #    Each row in temp_cases has a JSON array `cites_to`, and each entry
    #    in that array has its own `case_ids` array of cited cases:
    #
    #        data.cites_to = [
    #            { "case_ids": ["123", "456"], ... },
    #            { "case_ids": ["789"], ... },
    #        ]
    #
    #    LATERAL jsonb_array_elements flattens both levels:
    #       c1 -> cites_to_element -> case_ids   (one row per cited id)
    #
    #    We then INNER JOIN back to temp_cases (c2) on the cited id, which
    #    filters out citations pointing to cases outside our dataset - AGE
    #    would happily create dangling edges otherwise.
    #
    #    Edge creation is the slowest step (one cypher() call per edge), so
    #    we materialise the (from, to) pairs first, then loop in Python and
    #    print a progress line every PROGRESS_EVERY edges.
    # ------------------------------------------------------------------------
    cur.execute("""
        WITH edges AS (
            SELECT c1.data#>>'{id}' AS id_from, c2.data#>>'{id}' AS id_to
            FROM public.temp_cases c1
            LEFT JOIN
                LATERAL jsonb_array_elements(c1.data -> 'cites_to') AS cites_to_element ON true
            LEFT JOIN
                LATERAL jsonb_array_elements(cites_to_element -> 'case_ids') AS case_ids ON true
            JOIN public.temp_cases c2
                ON case_ids::text = c2.data#>>'{id}'
        )
        SELECT id_from::text, id_to::text FROM edges;
    """)
    # cur.fetchall() pulls the entire (id_from, id_to) result set into memory
    # as a Python list of tuples. The edge count is in the low thousands here,
    # so this is cheap - for a much larger graph you'd switch to a server-side
    # named cursor and stream rows instead.
    edge_pairs = cur.fetchall()
    total_edges = len(edge_pairs)
    print(f"Creating {total_edges} citation edges...")

    # PROGRESS_EVERY controls how often we print a progress line. We aim for
    # roughly 20 updates across the whole loop (one every ~5%). max(1, ...)
    # guards against the degenerate case where total_edges < 20, which would
    # otherwise make the divisor 0 and trigger a ZeroDivisionError on `i %`.
    PROGRESS_EVERY = max(1, total_edges // 20)

    # Capture a wall-clock start time so we can report an instantaneous
    # edges-per-second throughput alongside the percent-complete number.
    edge_start = time.time()

    # Loop over the materialised pairs and create one :REF edge per pair by
    # calling the PL/pgSQL helper defined in step 1b. enumerate(..., start=1)
    # gives us a 1-based counter so the progress output reads naturally
    # ("1/1234" rather than "0/1234").
    for i, (id_from, id_to) in enumerate(edge_pairs, start=1):
        cur.execute(
            "SELECT public.create_case_link_in_case_graph(%s, %s);",
            (id_from, id_to),
        )

        # Print progress on every Nth edge, and also on the very last edge
        # so the final "100.0%" line is guaranteed to appear even when
        # total_edges isn't a multiple of PROGRESS_EVERY.
        if i % PROGRESS_EVERY == 0 or i == total_edges:
            elapsed = time.time() - edge_start
            # Guard against divide-by-zero on the (very unlikely) case where
            # the first batch completes in under the timer's resolution.
            rate = i / elapsed if elapsed > 0 else 0
            pct = 100.0 * i / total_edges
            print(f"  edges: {i}/{total_edges} ({pct:.1f}%) - {rate:.1f} edges/s")

    print("Citation edges created.")

conn.close()
print("Done. Case graph is ready.")


In [ ]:
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_textsearch;")
    print("pg_textsearch extension is ready.")

    cur.execute("DROP INDEX IF EXISTS idx_cases_fts;")
    cur.execute("""
        CREATE INDEX idx_cases_fts
        ON public.cases
        USING bm25 (opinion)
        WITH (text_config = 'english');
        """)

    query = "water leaking"
    cur.execute("""
        SELECT id,
               name,
               opinion <@> to_bm25query(%s, 'idx_cases_fts') AS score
        FROM cases
        WHERE (opinion <@> to_bm25query(%s, 'idx_cases_fts')) < 0
        ORDER BY score
        LIMIT 5;
        """, (query, query))

    print("Top matches (lower BM25 scores are more relevant):")
    for case_id, name, score in cur.fetchall():
        print(f"  {case_id} | score={score:.3f} | {name}")

conn.close()


In [ ]:
with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_diskann;")

    # DiskANN index (quantization not supported yet on HorizonDB)
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_cases_diskann
        ON public.cases
        USING diskann (opinions_vector vector_cosine_ops)
        WITH (product_quantized='false');
    """)
print("pg_diskann ready: idx_cases_diskann on public.cases(opinions_vector)")